# **Few-shot protein-family generation with ProtGPT3-MSA**

**Notebook developed by Núria Mimbrero.**

In this practical we will use **seven confirmed defluorinase sequences** (a small group within the broader dehalogenase family) to prompt ProtGPT3-MSA. We will:

1. inspect the seven sequences we actually have;
2. explain why seven sequences are too few for sensible fine-tuning;
3. generate additional family-conditioned sequences without updating the model;
4. predict and display one structure; and
5. test copying and mutation recovery with 15 variants from the MegaScale stability dataset.

> Select **Runtime → Change runtime type → T4 GPU** before starting. Generated sequences and predicted structures are computational hypotheses, not experimentally validated proteins.


## **1. Setup**

We need only a language-model library, a sequence-alignment library, and a small 3D viewer.


In [ ]:
%pip install -q "transformers==4.53.3" accelerate biopython py3Dmol requests


**Mini-exercise.** Which package loads ProtGPT3-MSA, which one aligns protein sequences, and which one draws a structure?


In [ ]:
import requests
import torch
import py3Dmol
from Bio import Align
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Please select a GPU runtime and run again."
torch.manual_seed(7)
print(torch.cuda.get_device_name(0))


**Mini-exercise.** Change the random seed from `7` to another integer. Later, check whether the generated sequences change.


## **2. Start with the data we really have**

The original course example contains the following seven confirmed defluorinases. Keeping them in the notebook makes the practical self-contained and lets us see exactly what enters the model.


In [ ]:
defluorinases = [
    "MHAIKAVVFDLYGTLYDVYSVRTSCERIFPGQGEMVSKMWRQKQLEYTWMRTLMGQYQDFESATLDALRYTCGSLGLALDADGEAHLCSEYLSLTPFADVPQALQQLRAAGLKTAILSNGSRHSIRQVVGNSGLTNSFDHLISVDEVRLFKPHQKVYELAMDTLHLGESEILFVSCNSWDATGAKYFGYPVCWINRSNGVFDQLGVVPDIVVSDVGVLASRFSPVDEAA",
    "MEKPRALAPVPIRGVLFDAYGTLFDVYSVGLLAEQLFPGQGQTLGLLWRDKQIEYTRLVTTCNDGAHYQPFWDLTRSSLRYVCKRLALDLTPEREQRLMNQYRHLSAFPENKGVLQALKKRGIVTGILSNGDPSMLDVAVKSGGLEGLLDHVISVDSIRKYKTHPDAYALGPQHTGLDVRQIAFVSCNGWDALAATWYGYQTLWINRYQLPFEELGTAPTYTGSSLRDVLTLPGMNLPGAAA",
    "MAAAAGYSGERETAMIEAIAFDAYGTLYDVYSVERKCEEEYPGSGAAMSRLWRQKQLEYSWLRTLMGRYADFWRVTEDALRYTLAELGLEGDEGKIAAIMATYLELEMYPEVIEAMARFGTRKKAILTNGNLGMIQPLVARSGLGAHLDACLSADEAGLFKVRPEVYQLAVDHLGVARERLLFVSSNGWDVAGAKAFGFTVGWLNRRGLPPEELGVRADYEASNLLELAEMVVSA",
    "MIEAIAFDAYGTLYDVYSVERKCEEEYPGSGAAMSRLWRQKQLEYSWLRTLMGRYADFWRVTEDALRYTLAELGLEGDEGKIAAIMATYLELEMYPEVIEAMARFGTRKKAILTNGNLGMIEPLVARSGLDAHLDACLSADEAGLFKVRPEVYQLAVDHLGVARERLLFVSSNGWDVAGAKAFGFTVGWLNRRGLPPEELGVRADYEASNLLELTEMVVSA",
    "MHHSPKIKVLAFDIFGTVVDWHSSIVQEVKSLALNIDANQFALDWRAGYRPAMDQVLSGQQPWTSIDDIHRLILDELLAKYQISTLTEAQKMDLNFIWHRLNPWPDTVAALNQLKQDYIICTLSNGNIRLLVDLAKYAKLPWDTIFSAENFKAYKLSPKTYLGVSDFLNVAPSQVMMVATHQDDLAAARGCGLRTAYIERPFEYGAAQLKDSSPCIDNNLHATDLLNLVSLLKEKA",
    "MAGVPFRSPSTGRNVRAVLFDTFGTVVDWRTGIATAVADYAARHQLEVDAVAFADRWRARYQPSMDAILSGAREFVTLDILHRENLDFVLRESGIDPTNHDSGELDELARAWHVLTPWPDSVPGLTAIKAEYIIGPLSNGNTSLLLDMAKNAGIPWDVIIGSDINRKYKPDPQAYLRTAQVLGLHPGEVMLAAAHNGDLEAAHATGLATAFILRPVEHGPHQTDDLAPTGSWDISATDITDLAAQLRAGSTGFR",
    "MDVSNVRIVIFDTFGTVVNWHESVVQEGEALGRAKGVSIDWHEFANVWREEGYIKVMYEVAQGLRPWEPVDVLHRRKLDELLDVYGLKLTEEETDHFNRLWHRLLPWPDVQEGLRRLKTKYSIGPFSNGDFRLLLNMAKGSGLPWDFILAGQQFQKFKPDPTIYEDAVELLGGRPEEVLMVAAHPSDLDGAHAIGCPTLYVPRPLEYGAVNNHVEPEAKYDHETVADFRELAARLGV",
]


**Mini-exercise.** The list contains the complete input data. Before running the next cell, predict whether all seven proteins have the same length.


In [ ]:
print(f"We have only {len(defluorinases)} sequences.\n")
for number, sequence in enumerate(defluorinases, 1):
    print(f"Sequence {number} | {len(sequence)} aa\n{sequence}\n")


**Mini-exercise.** Add one line that prints the shortest and longest sequence lengths. Why might length variation matter during generation?


### **Why not fine-tune on seven sequences?**

Seven examples are far too few to estimate a family distribution by updating millions of model parameters. Fine-tuning would very likely **memorise these sequences** or overfit their accidental details.

ProtGPT3-MSA gives us another option: **few-shot prompting**. Its pretrained weights stay fixed. The seven homologues are placed in the context, and the model predicts an additional sequence that is compatible with that context.

This does not prove that a generated sequence is a defluorinase. It gives us family-conditioned candidates that still require computational and experimental validation.


## **3. Load ProtGPT3-MSA**


In [ ]:
model_id = "AI4PD/ProtGPT3-MSA"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(model_id, add_bos_token=False)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=dtype, device_map="auto"
).eval()


**Mini-exercise.** Print `model.config.num_hidden_layers`, `hidden_size`, and `num_local_experts`. What does each value describe?


## **4. Turn sequences into a prompt**

ProtGPT3-MSA expects four visible ingredients:

- `<|bos|>` marks the beginning;
- `1` requests N-to-C generation;
- `<no_gap>` says that these homologues are not aligned with gap characters; and
- `<s>` separates one protein from the next.

We deliberately keep the formatter short enough to inspect.


In [ ]:
def make_prompt(sequences):
    tokens = ["<|bos|>", "1", "<no_gap>"]
    for sequence in sequences:
        tokens += ["<s>", *sequence]
    return " ".join(tokens + ["<s>"])


**Mini-exercise.** For seven input proteins, how many `<s>` tokens should the prompt contain? Remember the final token asks the model to start one more sequence.


In [ ]:
prompt = make_prompt(defluorinases)
print(prompt[:300], "...")
print("<s> tokens:", prompt.count("<s>"))


**Mini-exercise.** Find the first three control tokens in the printed prompt. Which one would change if the sequences had already been aligned?


## **5. Generate one new family member**

The function below does one job: it generates residues until the next `<s>` boundary. `temperature` and `top_p` control sampling; they do not retrain the model.


In [ ]:
amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

@torch.inference_mode()
def generate_one(sequences, temperature=0.8, top_p=0.9):
    inputs = tokenizer(make_prompt(sequences), return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, do_sample=True, temperature=temperature, top_p=top_p,
        max_new_tokens=max(map(len, sequences)) + 50,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<s>")],
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).split("<s>")[0]
    return "".join(character for character in text if character in amino_acids)


**Mini-exercise.** Change `temperature` from `0.8` to `1.1`. Predict what should happen to diversity and to family similarity.


### **Why stop when the family reaches 16 sequences?**

ProtGPT3-MSA was trained on sets of **16 homologous sequences**. During inference we can supply at most 15 and ask for the 16th. Starting from seven natural sequences, the loop below adds nine candidates. Each new candidate becomes part of the next prompt.

Continuing beyond 16 would ask the model to operate outside the format it saw during training, so we stop deliberately.


In [ ]:
family = defluorinases.copy()
while len(family) < 16:
    family.append(generate_one(family))

generated = family[len(defluorinases):]
for number, sequence in enumerate(generated, 8):
    print(f"Sequence {number} | {len(sequence)} aa\n{sequence}\n")


**Mini-exercise.** Rerun from `family = defluorinases.copy()` with a different seed or temperature. Do all nine sequences change?


## **6. Predict and view one structure**

The public ESMFold service returns a **predicted** PDB for one candidate. A plausible-looking structure is useful for inspection, but it is not evidence of defluorinase activity or experimental stability.


In [ ]:
candidate = generated[0]
response = requests.post(
    "https://api.esmatlas.com/foldSequence/v1/pdb/", data=candidate, timeout=300
)
response.raise_for_status()
pdb = response.text
print(f"Predicted a structure for {len(candidate)} residues.")


**Mini-exercise.** Change `generated[0]` to another generated sequence. Is a sequence longer or shorter than the first necessarily more plausible?


In [ ]:
view = py3Dmol.view(width=800, height=500)
view.addModel(pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()


**Mini-exercise.** Replace `cartoon` with `stick`, or add a surface. Which representation makes the overall fold easiest to read?


## **7. Extra experiment: will the model copy 15 close variants?**

The [MegaScale dataset](https://huggingface.co/datasets/RosettaCommons/MegaScale) contains experimentally measured folding stabilities for many single-mutant variants. Downloading the full dataset during class would be wasteful, so we use one 47-residue design, `r10_437_TrROS_Hall.pdb`, and 15 mutation labels recorded in the dataset.

These are *variants of one scaffold*, not 15 independently evolved homologues. That makes this a useful stress test: will the model copy an input, reconstruct the common sequence, or introduce a new substitution?


In [ ]:
wild_type = "EPELVFKVRVRTKDGRELEIEVSAEDLEKLLEALPDIEEVEIEEVEP"
mutation_labels = [
    "T12F", "T12Y", "T12W", "E21W", "T12L", "E39W", "E21Y", "E39I",
    "I20F", "E17I", "E17W", "E17V", "P35G", "E28W", "V10W",
]

def apply_mutation(sequence, label):
    position = int(label[1:-1]) - 1
    assert sequence[position] == label[0]
    return sequence[:position] + label[-1] + sequence[position + 1:]

megascale = [apply_mutation(wild_type, label) for label in mutation_labels]


**Mini-exercise.** Interpret `T12F` in words. Then check why Python uses `- 1` when converting its position.


In [ ]:
for label, sequence in zip(mutation_labels, megascale):
    print(f"{label}: {sequence}")


**Mini-exercise.** Write a one-line check confirming that every variant differs from the wild type at exactly one position.


We now use all 15 variants as context. The generated protein is literally the **16th sequence** in the model's familiar training format.


In [ ]:
sequence_16 = generate_one(megascale)
print(sequence_16)
print("Exact copy of a prompt sequence:", sequence_16 in megascale)


**Mini-exercise.** Rerun this cell with a new seed. How often do you obtain an exact copy? Keep a tally across five runs.


In [ ]:
def mutations_from(reference, sequence):
    if len(reference) != len(sequence):
        return [f"length changed: {len(reference)} → {len(sequence)}"]
    return [f"{old}{i}{new}" for i, (old, new) in
            enumerate(zip(reference, sequence), 1) if old != new]

changes = mutations_from(wild_type, sequence_16)
print("Changes from the wild type:", changes or "none")
print("Changes absent from the prompt:", sorted(set(changes) - set(mutation_labels)))


**Mini-exercise.** Be precise about the word *new*: does it mean absent from these 15 prompts, absent from MegaScale, or never observed in nature? Which claim can this notebook support?


In [ ]:
aligner = Align.PairwiseAligner()
closest = max(megascale, key=lambda sequence: aligner.score(sequence, sequence_16))
print(aligner.align(closest, sequence_16)[0])


**Mini-exercise.** Compare the generated sequence with the wild type instead of the nearest prompt. Which comparison better answers the copying question?


## **Take-home messages**

- Seven sequences are useful as **context**, but not as a sensible fine-tuning dataset.
- ProtGPT3-MSA was trained on groups of 16, so we keep the prompt-plus-generation total at 16.
- A generated sequence can be an exact copy, a consensus-like reconstruction, or a new combination of substitutions. These outcomes should be measured, not assumed.
- A predicted PDB is a structural hypothesis. It does not validate function.

**Sources:** [ProtGPT3-MSA model card](https://huggingface.co/AI4PD/ProtGPT3-MSA), [MegaScale dataset](https://huggingface.co/datasets/RosettaCommons/MegaScale), and [ESMFold](https://doi.org/10.1126/science.ade2574).
